In [ ]:
import os
import pandas as pd
import sqlalchemy
import numpy as np
from sqlalchemy import text
from dotenv import load_dotenv

load_dotenv()  # .env 파일에서 DB 접속 정보 로드

engine = sqlalchemy.create_engine(
    "mysql+pymysql://{user}:{password}@{host}:{port}/{dbname}?charset=utf8".format(
        user     = os.getenv("DB_USER"),
        password = os.getenv("DB_PASSWORD"),
        host     = os.getenv("DB_HOST"),
        port     = os.getenv("DB_PORT"),
        dbname   = os.getenv("DB_NAME"),
    )
)

## 데이터 로드

In [ ]:
# returns: 반품 신청 내역 (사유 텍스트, 신청일, 취소일)
# members, orders와 JOIN하여 실제 구매자 + 결제 완료 건만 필터링
query = """
SELECT r.reason, r.memo, r.log_date, r.canceled_at
FROM returns AS r
INNER JOIN orders AS o ON r.order_id = o.order_id
INNER JOIN members AS m ON o.user_id = m.id
WHERE o.paid_at > '0000-00-00 00:00:00'
AND m.member_type != 'staff'
AND YEAR(r.log_date) != YEAR(r.canceled_at)
AND r.memo != '' AND r.memo != '-' AND r.memo != '.'
"""
df = pd.read_sql(query, engine)

In [ ]:
df['memo'].value_counts().head(20)

In [4]:
import re

def clean_reason(text):
    if pd.isna(text):
        return None
    text = text.lower()
    text = re.sub(r'\s+', '', text)      # 공백 제거
    text = re.sub(r'[^\w가-힣]', '', text)  # 특수문자 제거
    return text

In [5]:
df['memo_clean'] = df['memo'].apply(clean_reason)

In [ ]:
df[['memo', 'memo_clean']].tail(30)

In [7]:
CATEGORY_RULES = {
    '처방문제': ['처방안나옴', '처방', '수요', '환자', '병원'],
    '유효기간': ['유효', '유기', '유통', '기한', '기간', '날짜', '지남', '임박'],
    '제품불량': ['불량', '파손', '깨짐', '고장', '누출', '누수', '누액', '변질', '까짐', '변색', '회수', '용기', '불만', '컴플레인', '액이', 
            '부작용','깨진','손상', '불편함'],
    '폐업': ['폐업', '페업', '경영', '이전', '거래정지', '양수양도', '휴업'],
    '주문실수': ['실수', '바꿈', '재주문', '착오', '변경','새로','로 변경','잘못', '착각'],
    '재고과다': ['재고', '과다', '매출', '감소', '사용안함', '포화', '제고', '많아', '많이'],
    '판매부진': ['판매저조', '판매','저조', '주문부재', '단종', '부진', '안팔림', '필요가', '구매x', '어려움'],
    '단순변심': ['변심', '.', '-', '주문취소']
}

In [8]:
def map_category(reason):
    if not reason:
        return '기타'
    for category, keywords in CATEGORY_RULES.items():
        for kw in keywords:
            if kw in reason:
                return category
    return '기타'


In [9]:
df['memo_category'] = df['memo_clean'].apply(map_category)

In [10]:
reason_rank = (
    df.groupby('memo_category')
      .size()
      .reset_index(name='cnt')
      .sort_values('cnt', ascending=False)
)

In [ ]:
reason_rank

In [ ]:
df[df['memo_category'] == '기타']['memo'].value_counts().head(50)

In [ ]:
import plotly.express as px

# 막대 그래프 생성
fig = px.bar(reason_rank
    , x="memo_category"  # x축 데이터
    , y="cnt"  # y축 데이터
    #, color="col_3"  # 막대 색상 구분, 'col_3' 값에 따라 달라짐
    , title="반품 원인 분석"  # 그래프 제목 설정
    , barmode='group'  # 막대 표시 방식, 'group'은 나란히, 'stack'은 쌓아서 표시
)

# X축과 Y축 제목 설정 (선택적)
fig.update_layout(xaxis_title="반품 원인", yaxis_title="반품 수")

# 그래프 화면에 표시
fig.show()

In [33]:
# created_at 컬럼 있으면
df['month'] = pd.to_datetime(df['canceled_at']).dt.to_period('M')

In [ ]:
df[['canceled_at', 'month']].head()

In [39]:
monthly_reason = (
    df.groupby(['month', 'memo_category'])
      .size()
      .reset_index(name='cnt')
)

In [ ]:
monthly_reason.head()

In [45]:
pivot_df = monthly_reason.pivot(
    index='month',
    columns='memo_category',
    values='cnt'
).fillna(0)

In [ ]:
pivot_df

In [49]:
ratio_df = pivot_df.div(pivot_df.sum(axis=1), axis=0)

In [ ]:
ratio_df.head()

In [53]:
ratio_df.index = ratio_df.index.to_timestamp()

In [57]:
top_categories = (
    pivot_df.sum()
             .sort_values(ascending=False)
             .head(5)
             .index
)

ratio_top_df = ratio_df[top_categories]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(12, 6))

for col in ratio_top_df.columns:
    plt.plot(ratio_top_df.index, ratio_top_df[col], marker='o', label=col)

plt.title('월별 반품 사유 비중 변화 (TOP 5)')
plt.xlabel('월')
plt.ylabel('비중')

plt.legend(title='반품 사유', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [63]:
df['quarter'] = df['canceled_at'].dt.to_period('Q')

In [65]:
quarter_reason = (
    df.groupby(['quarter', 'memo_category'])
      .size()
      .reset_index(name='cnt')
)

In [67]:
pivot_q = quarter_reason.pivot(
    index='quarter',
    columns='memo_category',
    values='cnt'
).fillna(0)

ratio_q = pivot_q.div(pivot_q.sum(axis=1), axis=0)
ratio_q.index = ratio_q.index.to_timestamp()

In [ ]:
top_categories = (
    pivot_q.sum()
           .sort_values(ascending=False)
           .head(5)
           .index
)

plt.figure(figsize=(12, 6))
for col in top_categories:
    plt.plot(ratio_q.index, ratio_q[col], marker='o', label=col)

plt.title('분기별 반품 사유 비중 변화 (TOP 5)')
plt.xlabel('분기')
plt.ylabel('비중')

plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()